## Libraries
This section imports all the libraries used in this notebook.

In [1]:
import torch
import time
import torch.nn as nn
import torch.nn.functional as F
import math
import re
import textwrap
from collections import Counter


## Data Preparation
This section loads the book, removes the Project Gutenberg text at the
beginning and the end, and replaces the underscores used for italic words.

In [2]:
with open("data/sherlock.txt", encoding="utf-8") as f:
    text = f.read()

In [3]:
print(text[:1000])
print("=" * 50)
print(text[-1000:])

The Project Gutenberg eBook of The Adventures of Sherlock Holmes
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle


        
Release date: March 1, 1999 [eBook #1661]
                Most recently updated: October 10, 2023

Language: English

Other information and formats: www.gutenberg.org/ebooks/1661

Credits: an anonymous Project Gutenberg volunteer and Jose Menendez


*** START OF THE PROJECT GUTENBERG EBOOK THE ADVENTURES OF SHERLOCK HOLMES ***




The Adventures of Sherlock Holmes

by Arthur Conan Doyle


Contents

   I

In [4]:
print(text.find("Sherlock"))
print(text.find("*** START OF"))
print(text.find("*** END OF"))
print(len(text))

49
844
563136
581601


In [5]:
start = text.find("*** START OF")
start = text.find("\n", start)    #from the next line of start
end = text.find("*** END OF")     #stop at end sign

body = text[start:end]
body = body.strip()

#replace "_" as " "
body = body.replace("_", " ")

print(len(body))
print(body[:300])
print(body[-300:])
print("=" * 50)

562202
The Adventures of Sherlock Holmes

by Arthur Conan Doyle


Contents

   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of the 
ritius. As to Miss Violet
Hunter, my friend Holmes, rather to my disappointment, manifested no
further interest in her when once she had ceased to be the centre of
one of his problems, and she is now the head of a private school at
Walsall, where I believe that she has met with considerable success.


In [6]:
words = body.split()
print(len(body))
print(len(words))
print(len(set(words)))
print(words[:50])

562202
104565
14639
['The', 'Adventures', 'of', 'Sherlock', 'Holmes', 'by', 'Arthur', 'Conan', 'Doyle', 'Contents', 'I.', 'A', 'Scandal', 'in', 'Bohemia', 'II.', 'The', 'Red-Headed', 'League', 'III.', 'A', 'Case', 'of', 'Identity', 'IV.', 'The', 'Boscombe', 'Valley', 'Mystery', 'V.', 'The', 'Five', 'Orange', 'Pips', 'VI.', 'The', 'Man', 'with', 'the', 'Twisted', 'Lip', 'VII.', 'The', 'Adventure', 'of', 'the', 'Blue', 'Carbuncle', 'VIII.', 'The']


In [7]:
holmes_variants = [w for w in set(words) if w.lower().startswith("holmes")]
print(holmes_variants)
print(words[1000:1050])
print(len(holmes_variants))

['HOLMES,—Lord', 'HOLMES,—I', 'Holmes?', 'Holmes.”', 'HOLMES,—You', 'Holmes’', 'Holmes,', 'Holmes—a', 'Holmes!”', 'Holmes,”', 'Holmes.’”', 'Holmes', 'Holmes!', 'Holmes?”', 'Holmes.', 'Holmes;']
['of', 'the', 'sole', 'in', 'order', 'to', 'remove', 'crusted', 'mud', 'from', 'it.', 'Hence,', 'you', 'see,', 'my', 'double', 'deduction', 'that', 'you', 'had', 'been', 'out', 'in', 'vile', 'weather,', 'and', 'that', 'you', 'had', 'a', 'particularly', 'malignant', 'boot-slitting', 'specimen', 'of', 'the', 'London', 'slavey.', 'As', 'to', 'your', 'practice,', 'if', 'a', 'gentleman', 'walks', 'into', 'my', 'rooms', 'smelling']
16


In [8]:
#lowercase and split into words and punctuation
tokens = re.findall(r"\w+|[^\w\s]", body.lower())

print(len(tokens))
print(len(set(tokens)))

128736
7903


In [9]:
holmes_variants = [w for w in set(tokens) if w.lower().startswith("holmes")]
print(holmes_variants)
print(tokens[1000:1050])
print(len(holmes_variants))

['holmes']
['have', 'a', 'most', 'clumsy', 'and', 'careless', 'servant', 'girl', '?', '”', '“', 'my', 'dear', 'holmes', ',', '”', 'said', 'i', ',', '“', 'this', 'is', 'too', 'much', '.', 'you', 'would', 'certainly', 'have', 'been', 'burned', ',', 'had', 'you', 'lived', 'a', 'few', 'centuries', 'ago', '.', 'it', 'is', 'true', 'that', 'i', 'had', 'a', 'country', 'walk', 'on']
1


In [10]:
#print the frequency of tokens
counts = Counter(tokens)
print(counts.most_common(20))

#print how many words appear only once
once = [c for c in counts.values() if c == 1]
print(len(once))

#print how many words appear under 3 times
under3 = [c for c in counts.values() if c < 3]
print(len(under3))

#print how many words appear under 5 times
under5 = [c for c in counts.values() if c < 5]
print(len(under5))

[(',', 7659), ('.', 6208), ('the', 5630), ('i', 3038), ('and', 3018), ('“', 2764), ('to', 2744), ('of', 2655), ('a', 2642), ('”', 2325), ('in', 1766), ('that', 1752), ('it', 1734), ('you', 1502), ('he', 1483), ('was', 1410), ('his', 1159), ('is', 1124), ('’', 1051), ('my', 1007)]
3494
4762
5838


In [11]:
once_pect = round(sum(once) / len(tokens) * 100, 2)
print(once_pect)

under3_pect = round(sum(under3) / len(tokens) * 100, 2)
print(under3_pect)

under5_pect = round(sum(under5) / len(tokens) * 100, 2)
print(under5_pect)

2.71
4.68
7.5


## Vocabulary
This section keeps tokens that appear at least 3 times, replaces the
rest with <unk>, and turns every token into a number.

In [12]:
vocab = sorted([w for w, c in counts.items() if c >= 3])
vocab.append("<unk>")

In [13]:
stoi = {w: i for i, w in enumerate(vocab)}   # string to index
itos = {i: w for i, w in enumerate(vocab)}   # index to string

In [14]:
print(len(vocab))
print(vocab[:20])

3142
['!', '&', '(', ')', ',', '-', '.', '000', '10', '100', '1000', '11', '15', '1869', '2', '22nd', '30', '3rd', '4', '6']


In [15]:
ids  = [stoi.get(t, stoi["<unk>"])  for t in tokens]    # word → number
back = [itos[i] for i in ids[:20]]  # number → word
print(back)
print(tokens[:20])

['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur', '<unk>', '<unk>', 'contents', 'i', '.', 'a', 'scandal', 'in', 'bohemia', 'ii', '.', 'the', 'red']
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur', 'conan', 'doyle', 'contents', 'i', '.', 'a', 'scandal', 'in', 'bohemia', 'ii', '.', 'the', 'red']


In [16]:
data = torch.tensor(ids, dtype=torch.long)
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

In [17]:
print(len(train_data))
print(len(val_data))
print(len(train_data) + len(val_data))
print(len(data))

115862
12874
128736
128736


In [18]:
x = train_data[:12]
y = train_data[1:13]

for i in range(12):
    context = [itos[j.item()] for j in x[:i + 1]]
    target = itos[y[i].item()]

    print(context, "->", target)

['the'] -> adventures
['the', 'adventures'] -> of
['the', 'adventures', 'of'] -> sherlock
['the', 'adventures', 'of', 'sherlock'] -> holmes
['the', 'adventures', 'of', 'sherlock', 'holmes'] -> by
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by'] -> arthur
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur'] -> <unk>
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur', '<unk>'] -> <unk>
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur', '<unk>', '<unk>'] -> contents
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur', '<unk>', '<unk>', 'contents'] -> i
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur', '<unk>', '<unk>', 'contents', 'i'] -> .
['the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'arthur', '<unk>', '<unk>', 'contents', 'i', '.'] -> a


In [19]:
torch.manual_seed(412)

block_size = 16
batch_size = 4

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    
    return x, y

xb, yb = get_batch("train")
print(xb.shape)
print(yb.shape)
print([itos[j.item()] for j in xb[0]])
print([itos[j.item()] for j in yb[0]])

torch.Size([4, 16])
torch.Size([4, 16])
['been', 'able', 'to', '<unk>', ',', 'amount', 'to', '£', '<unk>', '10', 's', '.', ',', 'while', 'he', 'has']
['able', 'to', '<unk>', ',', 'amount', 'to', '£', '<unk>', '10', 's', '.', ',', 'while', 'he', 'has', '£']


## Training Setup
This section defines the shared functions for training, evaluating, and
generating text, so all three models are trained and tested the same way.

In [20]:
batch_size = 32
eval_interval = 500
eval_iters = 100

@torch.no_grad()    # no gradient tracking during evaluation
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(model, max_iters, lr):
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr)
    history = []
    start = time.time()
    for step in range(max_iters):
        if step % eval_interval == 0 or step == max_iters - 1:
            losses = estimate_loss(model)
            history.append((step,losses["train"], losses["val"]))
            print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none = True)
        loss.backward()
        optimizer.step()
    elapsed = time.time() - start
    return history, elapsed

@torch.no_grad()
def generate(model, start_text, max_new_tokens):
    model.eval()
    start_tokens = re.findall(r"\w+|[^\w\s]", start_text.lower())
    idx = torch.tensor([[stoi.get(t, stoi["<unk>"]) for t in start_tokens]], dtype=torch.long)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim = -1)
        next_id = torch.multinomial(probs, num_samples = 1)
        idx = torch.cat([idx, next_id], dim = 1)
    model.train()
    return " ".join(itos[i.item()] for i in idx[0])

## Model 1: Bigram
This section builds, trains, and tests the bigram model, which predicts
the next token using only the current token.

In [21]:
class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.table(idx)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

In [22]:
vocab_size = len(vocab)
model = BigramModel(vocab_size)

logits, loss = model(xb, yb)
print(logits.shape)
print(loss.item())
print(math.log(vocab_size))
print(sum(p.numel() for p in model.parameters()))

torch.Size([4, 16, 3142])
8.605178833007812
8.052614818815567
9872164


In [23]:
#train model
torch.manual_seed(412)
bigram = BigramModel(vocab_size)
bigram_history, bigram_time = train(bigram, max_iters = 3000, lr = 1e-2)
print(f"Training time: {bigram_time:.1f} seconds")

step 0: train 8.5629, val 8.5782
step 500: train 5.5459, val 5.9816
step 1000: train 4.6257, val 5.3699
step 1500: train 4.2393, val 5.2233
step 2000: train 4.0224, val 5.1736
step 2500: train 3.8700, val 5.1729
step 2999: train 3.7858, val 5.1560
Training time: 74.8 seconds


In [24]:
torch.manual_seed(412)
bigram_sample = generate(bigram, "holmes", 50)
print(textwrap.fill(bigram_sample, width = 58))

holmes lately throw tobacco family , mother had brought
some , laughing . did you an appointment whence wrist
could be useful boxes has a faded him back , if he has
been better for yourselves comfortable prisoner among the
proprietor backed bound from the mantelpiece error felt
that he rose


## Model 2: LSTM
This section builds, trains, and tests the LSTM model, which reads the
tokens in order and keeps a memory of the context.

In [25]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first = True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, idx, targets = None):
        x = self.embedding(idx)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B *T))
        return logits, loss

In [26]:
torch.manual_seed(412)
lstm = LSTMModel(vocab_size, embed_dim = 128, hidden_dim = 128)

logits, loss = lstm(xb, yb)
print(logits.shape)
print(loss.item())
print(sum(p.numel() for p in lstm.parameters()))

torch.Size([4, 16, 3142])
8.069317817687988
939590


In [27]:
torch.manual_seed(412)
lstm = LSTMModel(vocab_size, embed_dim = 128, hidden_dim = 128)
lstm_history, lstm_time = train(lstm, max_iters = 3000, lr = 3e-3)
print(f"Training time: {lstm_time:.1f} seconds")

step 0: train 8.0662, val 8.0644
step 500: train 4.2662, val 4.6316
step 1000: train 3.8865, val 4.5030
step 1500: train 3.6589, val 4.4759
step 2000: train 3.4563, val 4.5117
step 2500: train 3.2611, val 4.5770
step 2999: train 3.1295, val 4.6610
Training time: 35.0 seconds


In [28]:
torch.manual_seed(412)
lstm_sample = generate(lstm, "holmes", 50)
print(textwrap.fill(lstm_sample, width =80))

holmes , striking following his way in silence . i bowed with the vanishing
against the same position in which i has during the death and threw it , if he
has already not been torn at the border of tottenham court days from the
mantelpiece . “ there is not


## Model 3: Small Transformer
This section shows how the attention mask works, then builds, trains,
and tests a small transformer that uses self-attention.

In [29]:
T = 4
scores = torch.zeros(T, T)
mask = torch.tril(torch.ones(T, T))
print(mask)

scores = scores.masked_fill(mask == 0, float("-inf"))
print(scores)

weight = F.softmax(scores, dim = -1)
print(weight)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])
tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])


In [30]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        scores = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        weight = F.softmax(scores, dim = -1)
        v = self.value(x)
        return weight @ v

In [31]:
torch.manual_seed(412)
head = Head(n_embd = 128, head_size = 32)
x = torch.randn(4, 16, 128)
out = head(x)
print(out.shape)

torch.Size([4, 16, 32])


In [32]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        return self.proj(out)

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [33]:
torch.manual_seed(412)
block = Block(n_embd = 128, n_head = 4)
x = torch.randn(4, 16, 128)
print(block(x).shape)

torch.Size([4, 16, 128])


In [34]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets = None):
        B, T = idx.shape
        tok = self.token_embedding(idx)
        pos = self.position_embedding(torch.arange(T))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

In [35]:
torch.manual_seed(412)
transformer = TransformerModel(vocab_size, n_embd = 128, n_head = 4, n_layer = 2)

logits, loss = transformer(xb, yb)
print(logits.shape)
print(loss.item())
print(sum(p.numel() for p in transformer.parameters()))

torch.Size([4, 16, 3142])
8.192188262939453
1205574


In [36]:
torch.manual_seed(412)
transformer = TransformerModel(vocab_size, n_embd = 128, n_head = 4, n_layer = 2)
transformer_history, transformer_time = train(transformer, max_iters = 3000, lr = 1e-3)
print(f"Training time: {transformer_time:.1f} seconds")

step 0: train 8.2197, val 8.2212
step 500: train 4.3386, val 4.6376
step 1000: train 3.9308, val 4.5028
step 1500: train 3.6548, val 4.5018
step 2000: train 3.3889, val 4.5854
step 2500: train 3.1688, val 4.6477
step 2999: train 2.9465, val 4.8076
Training time: 55.0 seconds


In [37]:
torch.manual_seed(412)
transformer_sample = generate(transformer, "holmes", 50)
print(textwrap.fill(transformer_sample, width=80))

holmes turned up her family . ’ “ ‘ i say , ’ did you hot in what people could
be useful to you if and very <unk> , if you have used it clearer an <unk> <unk>
to your wishes , now , though a small felt <unk> shows which


## Experiment 2: Longer Context (block_size = 64)
This section trains all three models again with a longer context of 64
tokens, while keeping the number of tokens in each training step the same.

In [38]:
block_size = 64
batch_size = 8
print(block_size * batch_size)

512


In [39]:
torch.manual_seed(412)
bigram64 = BigramModel(vocab_size)
bigram64_history, bigram64_time = train(bigram64, max_iters = 3000, lr = 1e-2)
print(f"Training time: {bigram64_time:.1f} seconds")

step 0: train 8.5624, val 8.5761
step 500: train 5.5617, val 6.0038
step 1000: train 4.6764, val 5.3912
step 1500: train 4.2663, val 5.2380
step 2000: train 4.0296, val 5.1833
step 2500: train 3.8888, val 5.1314
step 2999: train 3.7869, val 5.1532
Training time: 75.5 seconds


In [40]:
torch.manual_seed(412)
lstm64 = LSTMModel(vocab_size, embed_dim = 128, hidden_dim = 128)
lstm64_history, lstm64_time = train(lstm64, max_iters = 3000, lr = 3e-3)
print(f"Training time: {lstm64_time:.1f} seconds")

step 0: train 8.0651, val 8.0646
step 500: train 4.2524, val 4.6336
step 1000: train 3.8568, val 4.4976
step 1500: train 3.6221, val 4.4693
step 2000: train 3.4198, val 4.4938
step 2500: train 3.2328, val 4.5365
step 2999: train 3.0651, val 4.5902
Training time: 45.0 seconds


In [41]:
torch.manual_seed(412)
transformer64 = TransformerModel(vocab_size, n_embd = 128, n_head = 4, n_layer = 2)
transformer64_history, transformer64_time = train(transformer64, max_iters = 3000, lr = 1e-3)
print(f"Training time: {transformer64_time:.1f} seconds")

step 0: train 8.2424, val 8.2291
step 500: train 4.5173, val 4.7877
step 1000: train 4.1347, val 4.5735
step 1500: train 3.8609, val 4.4914
step 2000: train 3.6071, val 4.4768
step 2500: train 3.4311, val 4.5198
step 2999: train 3.2142, val 4.6085
Training time: 67.2 seconds


## Generated Text (Experiment 2)
This section generates 50 tokens from each Experiment 2 model, starting
with "holmes".

In [42]:
torch.manual_seed(412)
bigram64_sample = generate(bigram64, "holmes", 50)
print(textwrap.fill(bigram64_sample, width=80))

holmes lately throw tobacco family , mother had brought some , laughing . did
you have carried out in that i should be deduced hollow distinguish him back ,
if he said he had drawn yourselves comfortable prisoner among the proprietor
backed bound from the mantelpiece error felt that he rose


In [43]:
torch.manual_seed(412)
lstm64_sample = generate(lstm64, "holmes", 50)
print(textwrap.fill(lstm64_sample, width=80))


holmes , striking following holmes , with dark . i bowed , “ did you see it ?
why could i know how you could distinguish him back , if he has already come to
an enemy . ’ “ ‘ some years ago , a murderous felt , what i


In [44]:

torch.manual_seed(412)
transformer64_sample = generate(transformer64, "holmes", 50)
print(textwrap.fill(transformer64_sample, width=80))

holmes , striking tobacco , for mother had brought in , with the county against
him in whence why that he has a baboon at and him back , if he has already done
for yourselves which he carried himself to the bell and under a member of his
own which


## Results
This section prints a table of the final loss, the best validation loss,
and the training time for all six models.

In [45]:
results = [
    ("Bigram (16)", bigram_history, bigram_time),
    ("LSTM (16)", lstm_history, lstm_time),
    ("Transformer (16)", transformer_history, transformer_time),
    ("Bigram (64)", bigram64_history, bigram64_time),
    ("LSTM (64)", lstm64_history, lstm64_time),
    ("Transformer (64)", transformer64_history, transformer64_time),
]

print(f"{'Model':<18}{'Train':>8}{'Val':>8}{'Best Val':>10}{'Best Step':>11}{'Time (s)':>10}")
for name, history, t in results:
    step, train_loss, val_loss = history[-1]
    best_step, _, best_val = min(history, key=lambda h: h[2])
    print(f"{name:<18}{train_loss:>8.2f}{val_loss:>8.2f}{best_val:>10.2f}{best_step:>11}{t:>10.1f}")

Model                Train     Val  Best Val  Best Step  Time (s)
Bigram (16)           3.79    5.16      5.16       2999      74.8
LSTM (16)             3.13    4.66      4.48       1500      35.0
Transformer (16)      2.95    4.81      4.50       1500      55.0
Bigram (64)           3.79    5.15      5.13       2500      75.5
LSTM (64)             3.07    4.59      4.47       1500      45.0
Transformer (64)      3.21    4.61      4.48       2000      67.2
